In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType
import uuid

df = spark.table("hack_nation.india_medical.india_facilities_raw")
print(f"Raw rows: {df.count()}")
df.printSchema()

Raw rows: 10033
root
 |-- name: string (nullable = true)
 |-- phone_numbers: string (nullable = true)
 |-- officialPhone: string (nullable = true)
 |-- email: string (nullable = true)
 |-- websites: string (nullable = true)
 |-- officialWebsite: string (nullable = true)
 |-- yearEstablished: string (nullable = true)
 |-- facebookLink: string (nullable = true)
 |-- twitterLink: string (nullable = true)
 |-- linkedinLink: string (nullable = true)
 |-- instagramLink: string (nullable = true)
 |-- address_line1: string (nullable = true)
 |-- address_line2: string (nullable = true)
 |-- address_line3: string (nullable = true)
 |-- address_city: string (nullable = true)
 |-- address_stateOrRegion: string (nullable = true)
 |-- address_zipOrPostcode: string (nullable = true)
 |-- address_country: string (nullable = true)
 |-- address_countryCode: string (nullable = true)
 |-- facilityTypeId: string (nullable = true)
 |-- operatorTypeId: string (nullable = true)
 |-- affiliationTypeIds: string

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

# Fix facilityTypeId typo
df = df.withColumn("facilityTypeId",
    F.when(F.col("facilityTypeId") == "farmacy", "pharmacy")
     .otherwise(F.col("facilityTypeId"))
)

# Cast lat/lng (these are clean)
df = df.withColumn("latitude",  F.col("latitude").cast(DoubleType()))
df = df.withColumn("longitude", F.col("longitude").cast(DoubleType()))

# Clean literal "null" strings BEFORE casting to INT
df = df.withColumn("numberDoctors",
    F.when(F.col("numberDoctors").isin("null", "NULL", "None", ""), None)
     .otherwise(F.col("numberDoctors"))
     .cast(IntegerType())
)
df = df.withColumn("capacity",
    F.when(F.col("capacity").isin("null", "NULL", "None", ""), None)
     .otherwise(F.col("capacity"))
     .cast(IntegerType())
)

# Also clean "null" strings in other text columns that may have them
for col_name in ["description", "operatorTypeId", "officialWebsite",
                 "email", "yearEstablished"]:
    df = df.withColumn(col_name,
        F.when(F.col(col_name).isin("null", "NULL", "None"), None)
         .otherwise(F.col(col_name))
    )

print("Cell 2 done")

Cell 2 done


In [0]:
df = df.withColumn("pin_code",
    F.when(
        F.col("address_zipOrPostcode").isin("null", "NULL", "None", ""), None
    ).otherwise(
        F.regexp_replace(F.col("address_zipOrPostcode"), r"\s+", "")
    )
)

# Flag invalid PINs (not exactly 6 digits)
df = df.withColumn("pin_code_valid",
    F.when(
        F.col("pin_code").isNull(), False
    ).when(
        F.col("pin_code").rlike(r"^\d{6}$"), True
    ).otherwise(False)
)

# Set non-valid PINs to null (keep the raw value in original column)
df = df.withColumn("pin_code",
    F.when(F.col("pin_code_valid"), F.col("pin_code")).otherwise(None)
)

In [0]:
STATE_MAP = {
    # Abbreviations
    "Ut": "Uttar Pradesh", "Up": "Uttar Pradesh", "U.p.": "Uttar Pradesh",
    "Gj": "Gujarat", "Mh": "Maharashtra", "Ka": "Karnataka",
    "Nct": "Delhi", "Ncr": "Delhi", "Nit": None,
    "J&k": "Jammu And Kashmir",
    # Misspellings
    "Tamilnadu": "Tamil Nadu", "Andhrapradesh": "Andhra Pradesh",
    "Madhyapradesh": "Madhya Pradesh", "Chattisgarh": "Chhattisgarh",
    "Uttaranchal": "Uttarakhand", "Pondicherry": "Puducherry",
    # Cities mistakenly put as state — Maharashtra
    "Nagpur": "Maharashtra", "Solapur": "Maharashtra", "Thane": "Maharashtra",
    "Beed": "Maharashtra", "Chandrapur": "Maharashtra", "Ambernath": "Maharashtra",
    "Mira Bhayander": "Maharashtra", "Chinchwad": "Maharashtra",
    "Pimpri-chinchwad": "Maharashtra", "Kalyan": "Maharashtra",
    "Navi Mumbai": "Maharashtra", "Navi Mumbai, Maharashtra": "Maharashtra",
    "Pune, Maharashtra": "Maharashtra", "Pune-411044": "Maharashtra",
    "Jalgaon District": "Maharashtra", "Durg": "Maharashtra",
    # Karnataka
    "Bangalore": "Karnataka", "Bengaluru": "Karnataka",
    "Belgaum": "Karnataka", "Udupi": "Karnataka", "Chikmagalur": "Karnataka",
    # Kerala
    "Kochi": "Kerala", "Ernakulam": "Kerala", "Thrissur": "Kerala",
    "Malappuram": "Kerala", "Malappuram, Kerala": "Kerala", "Kannur": "Kerala",
    "Pathanamthitta": "Kerala", "Palakkad": "Kerala", "Alappuzha": "Kerala",
    "Thiruvananthapuram": "Kerala", "Chittur": "Kerala",
    # Tamil Nadu
    "Thoothukudi": "Tamil Nadu", "Vellore": "Tamil Nadu", "Erode": "Tamil Nadu",
    "Salem": "Tamil Nadu", "Thanjavur": "Tamil Nadu", "Tiruvallur-602001": "Tamil Nadu",
    # Telangana
    "Hyderabad": "Telangana", "Secunderabad": "Telangana",
    "Karimnagar": "Telangana", "Telangana State": "Telangana", "Mandamarri": "Telangana",
    # Andhra Pradesh
    "Kurnool": "Andhra Pradesh", "Rajahmundry": "Andhra Pradesh",
    "Chittoor": "Andhra Pradesh", "Prakasam District": "Andhra Pradesh",
    # Haryana
    "Kurukshetra": "Haryana", "Gurugram": "Haryana", "Jhajjar": "Haryana",
    "Nuh": "Haryana", "Charkhi Dadri, Haryana": "Haryana",
    "Fatehabad, Haryana": "Haryana", "Sector 56": "Haryana",
    # Punjab
    "Amritsar": "Punjab", "Sangrur": "Punjab", "Gurdaspur": "Punjab",
    "Patiala": "Punjab", "Ludhiana": "Punjab", "Mohali": "Punjab",
    "Zirakpur": "Punjab", "Ropar": "Punjab", "Punjab Region": "Punjab",
    # Rajasthan
    "Jodhpur": "Rajasthan", "Jaipur": "Rajasthan", "Udaipur": "Rajasthan",
    "Sikar": "Rajasthan", "Churu": "Rajasthan", "Rajsamand, Rajasthan": "Rajasthan",
    "Pali-rajasthan": "Rajasthan", "Durgapura": "Rajasthan",
    # Gujarat
    "Rajkot": "Gujarat", "Surat": "Gujarat", "Mehsana": "Gujarat",
    "Bharuch": "Gujarat", "Gandhinagar": "Gujarat",
    "Surendranagar District": "Gujarat", "Veraval": "Gujarat",
    # Delhi
    "New Delhi": "Delhi", "Delhi Division": "Delhi", "Delhi Ncr": "Delhi",
    "North West Delhi": "Delhi", "West Delhi": "Delhi",
    "National Capital Territory Of Delhi": "Delhi", "Safdarjung Enclave": "Delhi",
    # Uttar Pradesh cities
    "Ghaziabad": "Uttar Pradesh", "Lucknow": "Uttar Pradesh",
    "Varanasi": "Uttar Pradesh", "Allahabad": "Uttar Pradesh",
    "Aligarh": "Uttar Pradesh", "Moradabad": "Uttar Pradesh",
    "Azamgarh": "Uttar Pradesh", "Ambedkar Nagar": "Uttar Pradesh",
    "Faizabad": "Uttar Pradesh", "Kalyanpur Kanpur": "Uttar Pradesh",
    "Gautam Buddha Nagar": "Uttar Pradesh",
    # West Bengal
    "Kolkata": "West Bengal", "Howrah": "West Bengal", "Hooghly": "West Bengal",
    "North 24 Parganas": "West Bengal", "Birbhum": "West Bengal",
    "Paschim Medinipur": "West Bengal", "Murshidabad": "West Bengal",
    "Alipurduar": "West Bengal", "Puruliya": "West Bengal",
    "Rajarhat": "West Bengal", "Durgapur": "West Bengal",
    "Chakdah": "West Bengal", "Kharagpur": "West Bengal", "Dinajpur": "West Bengal",
    "Khaira": "West Bengal",
    # Bihar
    "Gaya": "Bihar", "Saran": "Bihar", "Jehanabad, Bihar": "Bihar",
    "Sitamarhi": "Bihar", "Supaul": "Bihar", "Aurangabad-bihar": "Bihar",
    # Jharkhand
    "Bokaro": "Jharkhand", "Bokaro Steel City, Jharkhand": "Jharkhand",
    # Chhattisgarh
    "Raipur": "Chhattisgarh", "Bhilai": "Chhattisgarh", "Durg": "Chhattisgarh",
    # Madhya Pradesh
    "Singrauli": "Madhya Pradesh", "Jabalpur": "Madhya Pradesh",
    "Guna, Madhya Pradesh": "Madhya Pradesh",
    "Dhar District, Madhya Pradesh": "Madhya Pradesh", "Thatipur": "Madhya Pradesh",
    # Assam
    "Darrang": "Assam", "Golaghat": "Assam", "Silchar": "Assam",
    "Barpeta, Assam": "Assam", "Sibsagar": "Assam",
    # Uttarakhand
    "Mukteshwar": "Uttarakhand",
    # Jammu & Kashmir
    "Jammu & Kashmir": "Jammu And Kashmir", "Ganderbal": "Jammu And Kashmir",
    "Kupwara": "Jammu And Kashmir", "Anantnag": "Jammu And Kashmir",
    # Union territories
    "Daman And Diu": "Dadra And Nagar Haveli And Daman And Diu",
    "Ut Of Dadra & Nagar Haveli And Daman Diu": "Dadra And Nagar Haveli And Daman And Diu",
    # Tripura
    "West Tripura": "Tripura",
}

# Build a Spark CASE WHEN expression from the map
state_expr = F.col("address_stateOrRegion")
for dirty, clean in STATE_MAP.items():
    state_expr = F.when(
        F.col("address_stateOrRegion") == dirty,
        clean
    ).otherwise(state_expr)

df = df.withColumn("state_normalized", F.initcap(F.trim(state_expr)))

In [0]:
trust_expr = (
    F.lit(100)
    - F.when(
        (F.col("capability").contains("ICU") | F.col("capability").contains("Surgery") |
         F.col("capability").contains("Emergency")) &
        (F.col("equipment").isNull() | (F.col("equipment") == "[]")),
        F.lit(30)
    ).otherwise(F.lit(0))
    - F.when(
        F.col("capability").contains("Surgery") & F.col("numberDoctors").isNull(),
        F.lit(20)
    ).otherwise(F.lit(0))
    - F.when(
        F.col("description").isNull() &
        (F.col("capability").isNull() | (F.col("capability") == "[]")) &
        (F.col("procedure").isNull()  | (F.col("procedure")  == "[]")),
        F.lit(15)
    ).otherwise(F.lit(0))
    - F.when(
        F.col("specialties").isNull() | (F.col("specialties") == "[]"),
        F.lit(10)
    ).otherwise(F.lit(0))
).cast(IntegerType())

df = df.withColumn("trust_score", trust_expr)

df = df.withColumn("trust_flag",
    F.when(F.col("trust_score") >= 90, "VERIFIED")
     .when(F.col("trust_score") >= 70, "REVIEW")
     .otherwise("SUSPICIOUS")
)

In [0]:
def coalesce_text(*cols):
    return F.concat_ws(" | ", *[F.coalesce(F.col(c), F.lit("")) for c in cols])

df = df.withColumn("searchable_text",
    coalesce_text(
        "name", "description", "specialties",
        "procedure", "equipment", "capability",
        "address_city", "state_normalized", "pin_code", "facilityTypeId"
    )
)

In [0]:
df = df.withColumn("unique_id", F.monotonically_increasing_id().cast("string"))

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable("hack_nation.india_medical.india_facilities")

print(f"Written rows: {spark.table('hack_nation.india_medical.india_facilities').count()}")

---------------------------------------------------------------------------
NumberFormatException                     Traceback (most recent call last)
File <command-8697522434225982>, line 7
      1 df = df.withColumn("unique_id", F.monotonically_increasing_id().cast("string"))
      3 df.write \
      4     .format("delta") \
      5     .mode("overwrite") \
      6     .option("delta.enableChangeDataFeed", "true") \
----> 7     .saveAsTable("hack_nation.india_medical.india_facilities")
      9 print(f"Written rows: {spark.table('hack_nation.india_medical.india_facilities').count()}")

File /databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/readwriter.py:713, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    711 self._write.table_name = name
    712 self._write.table_save_method = "save_as_table"
--> 713 _, _, ei = self._spark.client.execute_command(
    714     self._write.command(self._spark.client), self._write.observations
    7